# RoBERTa SG Hate Speech Experiments

Three experiments comparing RoBERTa fine-tuning strategies for Singapore hate speech:

| Exp | Training data | Description |
|-----|--------------|-------------|
| **A** | SG only (~171 rows) | Train from scratch on SG data only |
| **B** | HateXplain → SG | Train on HateXplain first, then fine-tune on SG (catastrophic forgetting check) |
| **C** | HateXplain + SG combined | Train on mixed dataset |

All experiments evaluate on **both** test sets (HateXplain test + SG test).

In [6]:
!pip install -q transformers scikit-learn torch

## Setup & Config

In [7]:
import json
import os
import copy
import numpy as np
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME = "shareeep/roberta-base-fine-tuned-hatexplain"
NUM_LABELS = 3
MAX_LENGTH = 128

# ── Training ───────────────────────────────────────────────────────────────
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
WARMUP_RATIO  = 0.1
WEIGHT_DECAY  = 0.01

# Epochs per experiment (adjust as needed)
EPOCHS_EXP_A  = 20   # SG-only: small dataset, may need more epochs
EPOCHS_EXP_B_HATEX = 5  # HateXplain pre-training
EPOCHS_EXP_B_SG    = 10 # SG fine-tuning after HateXplain
EPOCHS_EXP_C  = 5    # Combined training

# ── Labels ─────────────────────────────────────────────────────────────────
LABEL_MAP   = {0: "hate", 1: "offensive", 2: "normal"}
LABEL_NAMES = ["hate", "offensive", "normal"]
SG_LABEL_TO_ID = {"hate": 0, "offensive": 1, "normal": 2}

Using device: cuda
GPU: Tesla T4


## Mount Drive / Set Paths
Toggle `USE_GOOGLE_DRIVE` depending on Colab vs local.

In [ ]:
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('Google Drive mounted')
    except Exception as e:
        print('Drive mount skipped:', e)
        USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    DRIVE_ROOT  = Path('/content/drive/MyDrive')
    DAP_HS_DIR  = DRIVE_ROOT / 'dap-hs'
    HATEX_DIR   = DRIVE_ROOT / 'hatexplain'
    OUTPUT_BASE = DAP_HS_DIR / 'roberta_sg_experiments'
else:
    # Local paths (relative to repo root)
    REPO_ROOT   = Path.cwd()
    DAP_HS_DIR  = REPO_ROOT
    HATEX_DIR   = REPO_ROOT / 'hatexplain'
    OUTPUT_BASE = REPO_ROOT / 'runs' / 'roberta_sg_experiments'

# ── HateXplain data ────────────────────────────────────────────────────────
HATEX_TRAIN = HATEX_DIR / 'hatexplain_train.jsonl'
HATEX_VAL   = HATEX_DIR / 'hatexplain_validation.jsonl'
HATEX_TEST  = HATEX_DIR / 'hatexplain_test.jsonl'

# ── SG synthetic data ─────────────────────────────────────────────────────
SG_TRAIN = DAP_HS_DIR / 'synthetic_training_20_terms_simplfied.jsonl'
SG_VAL   = DAP_HS_DIR / 'synthetic_val_20_terms_simplfied.jsonl'
SG_TEST  = DAP_HS_DIR / 'synthetic_test_20_terms_simplfied.jsonl'

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

print('HateXplain train:', HATEX_TRAIN, '| exists:', HATEX_TRAIN.exists())
print('HateXplain val:  ', HATEX_VAL,   '| exists:', HATEX_VAL.exists())
print('HateXplain test: ', HATEX_TEST,  '| exists:', HATEX_TEST.exists())
print('SG train:        ', SG_TRAIN,    '| exists:', SG_TRAIN.exists())
print('SG val:          ', SG_VAL,      '| exists:', SG_VAL.exists())
print('SG test:         ', SG_TEST,     '| exists:', SG_TEST.exists())
print('Output base:     ', OUTPUT_BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted
HateXplain train: /content/drive/MyDrive/hatexplain/train.jsonl | exists: True
HateXplain val:   /content/drive/MyDrive/hatexplain/valid.jsonl | exists: True
HateXplain test:  /content/drive/MyDrive/hatexplain/test.jsonl | exists: True
SG train:         /content/drive/MyDrive/dap-hs/synthetic_training_20_terms_simplfied.jsonl | exists: True
SG val:           /content/drive/MyDrive/dap-hs/synthetic_val_20_terms_simplfied.jsonl | exists: True
SG test:          /content/drive/MyDrive/dap-hs/synthetic_test_20_terms_simplfied.jsonl | exists: True
Output base:      /content/drive/MyDrive/dap-hs/roberta_sg_experiments


## Data Loading
Unified loader for both HateXplain (majority-vote) and SG (string labels) formats.

In [ ]:
def read_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def load_hatexplain(path):
    """Load HateXplain JSONL → (sentences, labels) with majority vote.
    Handles both raw format (post_tokens + annotators) and pre-processed (text + label).
    """
    data = read_jsonl(path)
    sentences, labels = [], []
    for sample in data:
        # Text: raw has post_tokens, pre-processed has text
        if 'post_tokens' in sample:
            sentences.append(' '.join(sample['post_tokens']))
        else:
            sentences.append(sample['text'])

        # Label: raw has annotators.label (majority vote), pre-processed has label directly
        if 'annotators' in sample:
            labels.append(Counter(sample['annotators']['label']).most_common(1)[0][0])
        else:
            lbl = sample['label']
            if isinstance(lbl, str):
                lbl = SG_LABEL_TO_ID.get(lbl, int(lbl))
            labels.append(lbl)
    return sentences, labels


def load_sg(path):
    """Load SG synthetic JSONL → (sentences, labels)."""
    data = read_jsonl(path)
    sentences = [row['text'] for row in data]
    labels = [SG_LABEL_TO_ID[row['label']] for row in data]
    return sentences, labels


# ── Load all splits ────────────────────────────────────────────────────────
hatex_train_s, hatex_train_l = load_hatexplain(HATEX_TRAIN)
hatex_val_s,   hatex_val_l   = load_hatexplain(HATEX_VAL)
hatex_test_s,  hatex_test_l  = load_hatexplain(HATEX_TEST)

sg_train_s, sg_train_l = load_sg(SG_TRAIN)
sg_val_s,   sg_val_l   = load_sg(SG_VAL)
sg_test_s,  sg_test_l  = load_sg(SG_TEST)

# ── Summary ────────────────────────────────────────────────────────────────
def print_split(name, sentences, labels):
    c = Counter(labels)
    total = len(labels)
    dist = {LABEL_MAP[k]: f"{v} ({v/total*100:.1f}%)" for k, v in sorted(c.items())}
    print(f"  {name:25s} {total:>6} rows | {dist}")

print('\nHateXplain:')
print_split('train', hatex_train_s, hatex_train_l)
print_split('val',   hatex_val_s,   hatex_val_l)
print_split('test',  hatex_test_s,  hatex_test_l)

print('\nSG Synthetic:')
print_split('train', sg_train_s, sg_train_l)
print_split('val',   sg_val_s,   sg_val_l)
print_split('test',  sg_test_s,  sg_test_l)

KeyError: 'post_tokens'

## Common Utilities
Shared dataset class, training loop, and evaluation.

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class HateSpeechDataset(Dataset):
    def __init__(self, sentences, labels, tokenizer, max_length=MAX_LENGTH):
        self.encodings = tokenizer(
            sentences, truncation=True, padding=True,
            max_length=max_length, return_tensors='pt',
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx],
        }


def make_loader(sentences, labels, batch_size=BATCH_SIZE, shuffle=False):
    ds = HateSpeechDataset(sentences, labels, tokenizer)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def load_fresh_model():
    """Load a fresh RoBERTa model with a new 3-class head."""
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True,
    )
    return model.to(device)


@torch.no_grad()
def evaluate(model, loader):
    """Run model on loader, return (preds, true_labels)."""
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return np.array(all_preds), np.array(all_labels)


def eval_and_report(model, loader, dataset_name):
    """Evaluate and print classification report."""
    preds, true = evaluate(model, loader)
    f1 = f1_score(true, preds, average='macro')
    print(f'\n{"=" * 60}')
    print(f'{dataset_name} — Macro F1: {f1:.4f}')
    print(f'{"=" * 60}')
    print(classification_report(true, preds, target_names=LABEL_NAMES))
    return f1


def train_model(
    model, train_loader, val_loader,
    num_epochs, checkpoint_dir,
    class_weights=None, lr=LEARNING_RATE,
):
    """Standard training loop with best-checkpoint saving."""
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    total_steps  = len(train_loader) * num_epochs
    warmup_steps = int(total_steps * WARMUP_RATIO)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps,
    )

    if class_weights is not None:
        loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float).to(device))
    else:
        loss_fn = nn.CrossEntropyLoss()

    best_val_f1 = 0.0
    best_epoch  = 0
    history = []

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for step, batch in enumerate(train_loader):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            optimizer.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss = loss_fn(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # Validate
        val_preds, val_true = evaluate(model, val_loader)
        val_f1 = f1_score(val_true, val_preds, average='macro')
        history.append({'epoch': epoch + 1, 'loss': avg_loss, 'val_f1': val_f1})

        print(f'  Epoch {epoch+1}/{num_epochs} — loss: {avg_loss:.4f} | val F1: {val_f1:.4f}', end='')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch  = epoch + 1
            model.save_pretrained(checkpoint_dir)
            tokenizer.save_pretrained(checkpoint_dir)
            print(f'  ★ saved', end='')

        print()

    print(f'\nBest epoch: {best_epoch} | Best val F1: {best_val_f1:.4f}')
    return history, best_val_f1


def compute_weights(labels):
    """Compute balanced class weights from label list."""
    return compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=labels)


print('✅ Utilities ready')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/852 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/358 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Utilities ready


## Build Data Loaders

In [14]:
# ── HateXplain loaders ─────────────────────────────────────────────────────
hatex_train_loader = make_loader(hatex_train_s, hatex_train_l, shuffle=True)
hatex_val_loader   = make_loader(hatex_val_s,   hatex_val_l)
hatex_test_loader  = make_loader(hatex_test_s,  hatex_test_l)

# ── SG loaders ──────────────────────────────────────────────────────────────
sg_train_loader = make_loader(sg_train_s, sg_train_l, shuffle=True)
sg_val_loader   = make_loader(sg_val_s,   sg_val_l)
sg_test_loader  = make_loader(sg_test_s,  sg_test_l)

# ── Combined loaders (Exp C) — oversample SG to ~10% of combined ──────────
import math
_oversample_ratio = max(1, math.ceil(len(hatex_train_s) * 0.1 / max(len(sg_train_s), 1)))
_sg_oversampled_s = sg_train_s * _oversample_ratio
_sg_oversampled_l = sg_train_l * _oversample_ratio

combined_train_s = hatex_train_s + _sg_oversampled_s
combined_train_l = hatex_train_l + _sg_oversampled_l
combined_train_loader = make_loader(combined_train_s, combined_train_l, shuffle=True)

# Use both val sets concatenated for combined validation
combined_val_s = hatex_val_s + sg_val_s
combined_val_l = hatex_val_l + sg_val_l

combined_val_loader = make_loader(combined_val_s, combined_val_l)
print(f'Combined    — train: {len(combined_train_loader)} batches ({len(combined_train_s)} rows) | val: {len(combined_val_loader)}')

print(f'SG oversample: {_oversample_ratio}x ({len(sg_train_s)} → {len(_sg_oversampled_s)} rows)')

print(f'HateXplain  — train: {len(hatex_train_loader)} batches | val: {len(hatex_val_loader)} | test: {len(hatex_test_loader)}')
print(f'SG          — train: {len(sg_train_loader)} batches | val: {len(sg_val_loader)} | test: {len(sg_test_loader)}')

NameError: name 'hatex_train_s' is not defined

---
## Experiment A: SG-Only Training
Train RoBERTa from scratch (fresh classifier head) using only the ~171 SG synthetic training rows. Evaluate on both test sets.

In [ ]:
print('=' * 60)
print('EXPERIMENT A: SG-Only Training')
print('=' * 60)

_n_sg = len(sg_train_s)
print(f'\nTraining data: {_n_sg} rows')
print(f'  HateXplain: 0 ({0:.1f}%)')
print(f'  SG:         {_n_sg} ({100:.1f}%)')
print()

model_a = load_fresh_model()
ckpt_a  = OUTPUT_BASE / 'exp_a_sg_only'

history_a, best_f1_a = train_model(
    model_a,
    train_loader=sg_train_loader,
    val_loader=sg_val_loader,
    num_epochs=EPOCHS_EXP_A,
    checkpoint_dir=ckpt_a,
    class_weights=compute_weights(sg_train_l),
)

In [ ]:
# Load best checkpoint and evaluate on both test sets
model_a = AutoModelForSequenceClassification.from_pretrained(ckpt_a).to(device)

f1_a_sg    = eval_and_report(model_a, sg_test_loader,    'Exp A → SG Test')
f1_a_hatex = eval_and_report(model_a, hatex_test_loader, 'Exp A → HateXplain Test')

---
## Experiment B: HateXplain → SG (Catastrophic Forgetting Check)
1. Load the pre-trained HateXplain model (`shareeep/roberta-base-fine-tuned-hatexplain`) — no need to retrain.
2. Continue fine-tuning that model on SG data.
3. Evaluate on both test sets to see if HateXplain performance degrades.

In [ ]:
print('=' * 60)
print('EXPERIMENT B — Phase 1: Load pre-trained HateXplain model')
print('=' * 60)
print(f'\nUsing pre-trained model: {MODEL_NAME}')
print('(Already fine-tuned on HateXplain — skipping re-training)')

# Load the pre-trained model directly (no fresh head, no retraining needed)
model_b = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS,
)
model_b = model_b.to(device)

# Save as checkpoint so Phase 2 can load from it
ckpt_b_hatex = OUTPUT_BASE / 'exp_b_hatex_pretrained'
ckpt_b_hatex.mkdir(parents=True, exist_ok=True)
model_b.save_pretrained(ckpt_b_hatex)
tokenizer.save_pretrained(ckpt_b_hatex)

print(f'Saved to: {ckpt_b_hatex}')

In [ ]:
# Baseline: evaluate HateXplain-trained model on both test sets BEFORE SG fine-tuning
model_b = AutoModelForSequenceClassification.from_pretrained(ckpt_b_hatex).to(device)

print('--- Before SG fine-tuning ---')
f1_b_hatex_before = eval_and_report(model_b, hatex_test_loader, 'Exp B (before SG) → HateXplain Test')
f1_b_sg_before    = eval_and_report(model_b, sg_test_loader,    'Exp B (before SG) → SG Test')

In [ ]:
print('=' * 60)
print('EXPERIMENT B — Phase 2: Fine-tune on SG data')
print('=' * 60)

_n_sg = len(sg_train_s)
print(f'\nPhase 2 training data: {_n_sg} rows')
print(f'  HateXplain: 0 ({0:.1f}%)')
print(f'  SG:         {_n_sg} ({100:.1f}%)')
print(f'  (starting from HateXplain-pretrained checkpoint)')
print()

# Load the HateXplain-trained checkpoint and continue training on SG
model_b = AutoModelForSequenceClassification.from_pretrained(ckpt_b_hatex).to(device)
ckpt_b_sg = OUTPUT_BASE / 'exp_b_hatex_then_sg'

history_b2, best_f1_b2 = train_model(
    model_b,
    train_loader=sg_train_loader,
    val_loader=sg_val_loader,
    num_epochs=EPOCHS_EXP_B_SG,
    checkpoint_dir=ckpt_b_sg,
    class_weights=compute_weights(sg_train_l),
)

In [ ]:
# Load best SG-finetuned checkpoint and evaluate on both test sets
model_b = AutoModelForSequenceClassification.from_pretrained(ckpt_b_sg).to(device)

print('--- After SG fine-tuning ---')
f1_b_sg_after    = eval_and_report(model_b, sg_test_loader,    'Exp B (after SG) → SG Test')
f1_b_hatex_after = eval_and_report(model_b, hatex_test_loader, 'Exp B (after SG) → HateXplain Test')

print(f'\n--- Catastrophic Forgetting Check ---')
print(f'HateXplain F1 before SG: {f1_b_hatex_before:.4f}')
print(f'HateXplain F1 after SG:  {f1_b_hatex_after:.4f}')
print(f'Delta:                   {f1_b_hatex_after - f1_b_hatex_before:+.4f}')

---
## Experiment C: Combined Training (HateXplain + SG, oversampled)
Train on the merged training set. SG data is oversampled so it's ~10% of the combined set instead of ~1%.

In [ ]:
print('=' * 60)
print('EXPERIMENT C: Combined Training (HateXplain + SG)')
print('=' * 60)

_n_hatex = len(hatex_train_s)
_n_sg_orig = len(sg_train_s)
_n_sg_over = len(_sg_oversampled_s)
_n_total = _n_hatex + _n_sg_over
print(f'\nTraining data: {_n_total} rows')
print(f'  HateXplain: {_n_hatex} ({_n_hatex/_n_total*100:.1f}%)')
print(f'  SG:         {_n_sg_over} ({_n_sg_over/_n_total*100:.1f}%) — oversampled {_oversample_ratio}x from {_n_sg_orig}')
print()

model_c = load_fresh_model()
ckpt_c  = OUTPUT_BASE / 'exp_c_combined'

history_c, best_f1_c = train_model(
    model_c,
    train_loader=combined_train_loader,
    val_loader=combined_val_loader,
    num_epochs=EPOCHS_EXP_C,
    checkpoint_dir=ckpt_c,
    class_weights=compute_weights(combined_train_l),
)

In [ ]:
# Load best checkpoint and evaluate on both test sets
model_c = AutoModelForSequenceClassification.from_pretrained(ckpt_c).to(device)

f1_c_sg    = eval_and_report(model_c, sg_test_loader,    'Exp C → SG Test')
f1_c_hatex = eval_and_report(model_c, hatex_test_loader, 'Exp C → HateXplain Test')

---
## Results Summary

In [ ]:
print('\n' + '=' * 70)
print('RESULTS SUMMARY — Macro F1 Scores')
print('=' * 70)
print(f'{"Experiment":<45} {"SG Test":>10} {"HateX Test":>12}')
print('-' * 70)
print(f'{"A: SG-only":<45} {f1_a_sg:>10.4f} {f1_a_hatex:>12.4f}')
print(f'{"B: HateXplain-pretrained (before SG ft)":<45} {f1_b_sg_before:>10.4f} {f1_b_hatex_before:>12.4f}')
print(f'{"B: HateXplain → SG (after SG ft)":<45} {f1_b_sg_after:>10.4f} {f1_b_hatex_after:>12.4f}')
print(f'{"C: Combined (HateXplain + SG)":<45} {f1_c_sg:>10.4f} {f1_c_hatex:>12.4f}')
print('-' * 70)
print(f'\nCatastrophic forgetting (Exp B):')
print(f'  HateXplain F1 drop after SG fine-tuning: {f1_b_hatex_after - f1_b_hatex_before:+.4f}')

## Quick Predict

In [ ]:
def predict(model, sentence):
    model.eval()
    inputs = tokenizer(
        sentence, return_tensors='pt', truncation=True,
        max_length=MAX_LENGTH, padding=True,
    ).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=1)[0].cpu().numpy()
    pred_idx = int(np.argmax(probs))
    return {
        'label': LABEL_MAP[pred_idx],
        'confidence': round(float(probs[pred_idx]), 4),
        'probs': {LABEL_MAP[i]: round(float(p), 4) for i, p in enumerate(probs)},
    }


# Load whichever model you want to test
# model_test = AutoModelForSequenceClassification.from_pretrained(ckpt_a).to(device)  # Exp A
# model_test = AutoModelForSequenceClassification.from_pretrained(ckpt_b_sg).to(device)  # Exp B
model_test = AutoModelForSequenceClassification.from_pretrained(ckpt_c).to(device)  # Exp C

examples = [
    "All those cina babi should go back to China.",
    "That keling guy at the kopitiam damn annoying.",
    "Wah this ah kua very funny lah.",
    "The new hawker centre food is amazing.",
    "You are literally the dumbest person on the internet.",
    "We should get rid of every single foreigner in this country.",
]

for s in examples:
    r = predict(model_test, s)
    print(f"\n{s}")
    print(f"  → {r['label'].upper()} ({r['confidence']*100:.1f}%) | {r['probs']}")